## Install Packages and Set Up Path

In [ ]:
"""
This Section sets up the environment by installing required packages from requirements.txt and configuring the PYTHONPATH 
to include the infrapy package path.
"""
import sys, os
print('Installing requirements from ../requirements.txt (this may take a few minutes)')
# Preferred in-notebook installer
get_ipython()
%pip install -r ../requirements.txt
# Set up PYTHONPATH to include infrapy path (Update Path Below)
local_infrapy_path = r'C:\Users\ISLA-ENG-01\Documents\Projects\infrapy-ISLA'
if local_infrapy_path not in sys.path:
    sys.path.insert(0, local_infrapy_path)
    os.environ['PYTHONPATH'] = local_infrapy_path
print(f'PYTHONPATH set to {local_infrapy_path} \n Environment is now ready')

## Load in Variables from Configuration File


In [ ]:
"""
This section pulls in a config file and loads the necessary parameters for beamforming and detection using Infrapy's built-in config parser.

"""

import configparser

# InfraPy imports (use functions directly from the installed/source package)
from infrapy.utils import config as infraconfig

user_config = configparser.ConfigParser()
user_config.read('../config/example.ini')

# Helper to pull params from InfraPy user config
def cfg(param_section, param_name, dtype='float', cli_val=None):
    return infraconfig.set_param(user_config, param_section, param_name, cli_val, dtype)

# beamforming parameters
freq_min = cfg('FK', 'freq_min', 'float') 
freq_max = cfg('FK', 'freq_max', 'float') 
back_az_min = cfg('FK', 'back_az_min', 'float') 
back_az_max = cfg('FK', 'back_az_max', 'float') 
back_az_step = cfg('FK', 'back_az_step', 'float') 
trace_vel_min = cfg('FK', 'trace_vel_min', 'float') 
trace_vel_max = cfg('FK', 'trace_vel_max', 'float') 
trace_vel_step = cfg('FK', 'trace_vel_step', 'float') 
method = infraconfig.set_param(user_config, 'FK', 'method', None, 'string') 
signal_start = infraconfig.set_param(user_config, 'FK', 'signal_start', None, 'string')
signal_end = infraconfig.set_param(user_config, 'FK', 'signal_end', None, 'string')
noise_start = infraconfig.set_param(user_config, 'FK', 'noise_start', None, 'string')
noise_end = infraconfig.set_param(user_config, 'FK', 'noise_end', None, 'string')
window_len = cfg('FK', 'window_len', 'float') 
sub_window_len = cfg('FK', 'sub_window_len', 'float')
window_step = cfg('FK', 'window_step', 'float') 
cpu_cnt = infraconfig.set_param(user_config, 'FK', 'cpu_cnt', None, 'int')

# detection parameters
fd_window_len = cfg('FD', 'window_len', 'float') 
p_value = cfg('FD', 'p_value', 'float') 
min_duration = cfg('FD', 'min_duration', 'float') 
back_az_width = cfg('FD', 'back_az_width', 'float') 
fixed_thresh = cfg('FD', 'fixed_thresh', 'float')
thresh_ceil = cfg('FD', 'thresh_ceil', 'float')
return_thresh = infraconfig.set_param(user_config, 'FD', 'return_thresh', None, 'bool') or False
# NOTE: Merge detections currently needs to be improved. Should investigate how they associate nearby detections (time window, etc).
merge_dets = infraconfig.set_param(user_config, 'FD', 'merge_dets', None, 'bool') or True

## Run Beamforming and Detection Algorithm on Seedlink

In [ ]:
"""
This section runs an automated infrasonic detection using InfraPy's beamforming and detection modules. 
It will pull data from either IRIS or a local seedlink server, process it in overlapping time windows, and save detections 
and raw data to specified directories. Please update them to match the intented directories.

"""
import obspy
from obspy.core.util import AttribDict
from obspy.clients.fdsn import Client
from obspy.clients.seedlink import Client as Client_seedlink
import numpy as np
import time
import json
import os
# InfraPy imports
from infrapy.utils import data_io
from infrapy.detection import beamforming_new as fkd

wf_client = 1 # Flag to pull data from IRIS (0) or seedlink (1)
real_time = 1 # Flag to for static time frame (0) or real-time processing (1)
i = 0
# Currently using while loop for simplicity, i < 71 corresponds with ~24 hours w/ currently set up at 10 minute intervals with 1.5 minutes of overlap.
# For the current setup it takes about 2.25 minutes to run 10 minutes at .5-8 Hz range.
while i < 600:  
    stop_watch = time.time()
    # Adding event params
    EVENT_CONFIG = {
        'name': 'auto_infrapy_test',
        'network': 'IM',
        'station': 'I59*',
        'location': '',
        'channel': 'BDF',
        'start_time': obspy.UTCDateTime()-720 if(real_time) else obspy.UTCDateTime('2025-12-10T18:30:00.000000Z'),
        'end_time': obspy.UTCDateTime()-120 if(real_time) else obspy.UTCDateTime('2025-12-10T19:00:00.000000Z'),
    }

    # Set parameters from the event config
    name = EVENT_CONFIG['name']
    network = EVENT_CONFIG['network']
    station = EVENT_CONFIG['station']
    location = EVENT_CONFIG['location']
    channel = EVENT_CONFIG['channel']
    if (not i):
        t1 = EVENT_CONFIG['start_time']-480
    else:
        t1 = EVENT_CONFIG['start_time']
    t2 = EVENT_CONFIG['end_time']

    # Get waveforms from IRIS or seedlink
    if i == 0:
        try:
            client = Client('IRIS')
            inventory = client.get_stations(network=network,
                                            station=station,
                                            location=location,
                                            channel=channel,
                                            starttime=t1,
                                            endtime=t2,
                                            level="response")


        except:
            print('Error fetching data from FDSN client. Please check network/station codes and time range.')
            break

    # Set up seedlink and take in stream
    LOCAL_SEEDLINK = "192.168.112.200"

    seed = Client_seedlink(LOCAL_SEEDLINK, port=18000, timeout=180)
    try:
        if(wf_client):
            g_stream = seed.get_waveforms(network=network, location=location, station=station,
                                    channel=channel, starttime=t1, endtime=t2)
            if len(g_stream) > 0:
                print("Data Found on seedlink")
            else:
                print("Error fetching data from Seedlink. WiFi is correct, possibly an issue with retrieving data from CTBTO.")
        else:
            g_stream = client.get_waveforms(network=network, location=location, station=station,
                                    channel=channel, starttime=t1, endtime=t2)
            if len(g_stream) > 0:
                print("Data Found on IRIS")
    except:
        print("Error fetching data")
    # Add coordinates to stream using inv feteched from IRIS
    latlon = []
    for tr in g_stream:
        coords = inventory.get_coordinates(f"{network}.{tr.stats.station}.{location}.{channel}", t1)
        tr.stats.coordinates = AttribDict({
        'latitude': coords['latitude'],
        'elevation': coords['elevation'],
        'longitude': coords['longitude']})
        latlon.append((coords['latitude'], coords['longitude']))
        print(tr.stats.starttime, tr.stats.station)
    print(f'Fetched {len(g_stream)} traces from {network}.{station}.')

    # Get the centroid of the array. Standard coords are fine bc array isn't big enough for geodeisic shifting to occur.
    centroid = np.mean([lat for lat, lon in latlon]), np.mean([lon for lat, lon in latlon])
    array_lat, array_lon = centroid

    strm = g_stream.copy()
    print(f'Run iteration {i}')

    # Noise is calculated based on the previous stream. If the previous stream has detections it wil use the most current stream that does not have any detections.
    if (not i):
        # i==0 is a special case in which the previous 8 minutes of signal as the baseline noise.
        prev_start_time = t1
        dets = 0
        noise_start = t1
        noise_end = t1 + 480
        strm = strm.trim(t1+480, t2)
        n_strm = g_stream.trim(t1, t1+480)
    else:
        if(not dets):
            noise_start = prev_start_time
            noise_end = prev_start_time + 480
        else:
            pass
    
    print(f"Noise window: {noise_start} to {noise_end}")
    # Compute noise and signal indices in seconds relative to stream start (t1).
    noise_len = (noise_end - noise_start)
    subset_start = t1 if(i) else t1 + 480 
    str_name = name + "_" + t1.strftime("%Y%m%d_%H%M%S")
    print(f"Running Detection {subset_start} to {t2}")
    print(f"Processing stream: {str_name}")


    # Setup bf inputs based on config file
    back_az_vals = np.arange(back_az_min, back_az_max, back_az_step)
    trc_vel_vals = np.arange(trace_vel_min, trace_vel_max, trace_vel_step)
    
    # Run beamforming
    print(f"Running {method} beamforming")

    x, t, t0, geom = fkd.stream_to_array_data(strm, latlon=latlon)
    M, N = x.shape

    slowness = fkd.build_slowness(back_az_vals, trc_vel_vals)
    delays = fkd.compute_delays(geom, slowness)

    # Beamforming returns beam_power as a 3D array. Need to look into what actually is returned and best way to access this data
    beam_times, beam_peaks, beam_power = fkd.auto_run_bf(
                                                    (subset_start - t1),
                                                    (t2 - subset_start), freq_band=[freq_min, freq_max],
                                                    window_len=window_len,
                                                    sub_window_len=sub_window_len,
                                                    window_step=window_step,
                                                    method=method,
                                                    back_az_vals=back_az_vals,
                                                    trc_vel_vals=trc_vel_vals,
                                                    array_data=[x, t, t0, geom],
                                                    delays=delays
                                                    )

    # Run detection
    print(f"Running FD detection")
    # Compute noise _fstat for detection auto threshold ; IPBeamformingWidget.py lines 1402 -> 1449 
    TB_prod = (freq_max - freq_min) * window_len
    if(fixed_thresh):
        thresh = fixed_thresh
    else:
        # 
        # If detections were found thresh will be the same as previous valid fstat threshold. If not recompute with the previous timeslot
        if (not i):
            n_x, n_t, n_t0, n_geom = fkd.stream_to_array_data(n_strm, latlon=latlon)
            n_delays = fkd.compute_delays(n_geom, slowness)
            thresh = fkd.adjust_thresh_noise([n_x, n_t, n_t0, n_geom], window_len, sub_window_len, noise_len, window_step, freq_min, freq_max, method, back_az_vals, trc_vel_vals, n_delays, p_value, TB_prod)
        elif (dets):
            thresh = prev_thresh
        else:
            thresh = new_thresh
            
    min_seq = int(max(2, min_duration / (window_step)))
    det_results = fkd.run_fd(beam_times, beam_peaks, window_len, TB_prod, len(strm), 
                      p_value, min_seq, back_az_width, thresh, thresh_ceil, 
                      return_thresh, merge_dets)
    dets = det_results[0] if return_thresh else det_results
    
    det_list = []
    for det_info in dets:
        det = data_io.define_detection(det_info, [array_lat, array_lon], len(strm), 
                                      [freq_min, freq_max], note="Automated run", method=method)
        det_list.append(det)
    

    print(f"Detection Complete")

# Save detections
    if len(det_list) > 0:
        det_fpath = "../results/" + t1.strftime("%Y/%m/%d/")
        try:
            if not os.path.isdir(det_fpath):
                print("Making New Folder")
                os.makedirs(det_fpath, exist_ok=True)
            else:
                pass
        except:
            print("Error making folder, please investigate issues")
            det_fpath = "../results/bin/"


        det_out = det_fpath + str_name + '_detections.json'
        dets = 1
        prev_thresh = thresh
        print(f"  Found {len(det_list)} detections, writing to {det_out}\nNew Threshold: {prev_thresh}")
        str_info = [strm[0].stats.network,
                       strm[0].stats.station + "-" + strm[-1].stats.station,
                       strm[0].stats.channel]
        data_io.detection_list_to_json(det_out, det_list, str_info)
        
        with open(det_out, 'r') as f:
            dets_data = json.load(f)
        
        for det in dets_data:
            det['Name'] = str_name
            det['Latitude'] = array_lat
            det['Longitude'] = array_lon
            det['Signal'] = f"{subset_start} to {t2}"
            det['Noise'] = f"{noise_start} to {noise_end}"
            det['F-Stat Threshold'] = thresh

        
        with open(det_out, 'w') as f:
            json.dump(dets_data, f, indent=4)

        # If there is a detection save off all raw data
        dt = np.array([(tn - np.datetime64(strm[0].stats.starttime)).astype('m8[ms]').astype(float) * 1.0e-3 for tn in beam_times])
        raw_data = np.hstack((np.atleast_2d(dt).T, beam_peaks))
        rd_header = data_io.fk_header(strm, latlon, freq_min, freq_max, back_az_min, back_az_max, 
                                  back_az_step, trace_vel_min, trace_vel_max, trace_vel_step, 
                                  method, subset_start, t2, noise_start, noise_end, 
                                  window_len, sub_window_len, window_step)
    
        rd_out = det_fpath + str_name + '_raw_data.txt'
        np.savetxt(rd_out, raw_data, header=rd_header)

        # Copilot add code below here
        


        # Copilot add above below here

        
        print(f"  Wrote FK results to {rd_out}")
    else:
        print("No detections found.")
        dets = 0
        prev_start_time = subset_start
        new_thresh = fkd.adjust_thresh_noise([x, t, t0, geom], window_len, sub_window_len, noise_len, window_step, freq_min, freq_max, method, back_az_vals, trc_vel_vals, delays, p_value, TB_prod)
    if(real_time):
        T = time.time() - stop_watch
        print(f"Sleeping for {510-T} seconds until {obspy.UTCDateTime()+(510-T)-36000} (HST)")
        time.sleep(510-T)
    i += 1
    

## Create Visualizations for Detections

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from pathlib import Path
import obspy
from obspy.imaging.cm import obspy_sequential
labels = ['f-stat', 'baz', 'slow']
xlocator = mdates.AutoDateLocator()


# Set the FK raw data file you want to inspect
fpath = Path("..\\results\\12\\15\\auto_infrapy_test_20251215_224700_raw_data.txt")

# Load numeric columns (skip header lines starting with '#')
raw = np.genfromtxt(fpath, comments="#")

# Extract columns
seconds = raw[:, 0]
f_stat = raw[:, 1]
back_az = raw[:, 2]
trace_vel = raw[:, 3]

out = np.column_stack((seconds, f_stat, back_az, trace_vel))

print(f"Loaded {raw.shape[0]} rows from {fpath}")
print("First 5 rows:")
print(raw[:5])

fig = plt.figure()
for i, lab in enumerate(labels):
    ax = fig.add_subplot(4, 1, i + 1)
    ax.scatter(out[:, 0], out[:, i + 1], c=out[:, 1], alpha=0.6,
               edgecolors='none', cmap=obspy_sequential)
    ax.set_ylabel(lab)
    ax.set_xlim(out[0, 0], out[-1, 0])
    ax.set_ylim(out[:, i + 1].min(), out[:, i + 1].max())
    ax.xaxis.set_major_locator(xlocator)
    ax.xaxis.set_major_formatter(mdates.AutoDateFormatter(xlocator)))
fig.autofmt_xdate()
fig.subplots_adjust(left=0.15, top=0.95, right=0.95, bottom=0.2, hspace=0)
plt.show()